# Tokenize, embed, and index into Vector Search

Runs **inside Databricks**. Run `import_to_delta.ipynb` first — this notebook
reads the `listing_features`, `review_text`, and `listing_proximity` Delta
tables it creates.

**What it does**
1. Builds two kinds of text document per listing:
   - `listing` — name, type, location, description, amenities ("what the host says")
   - `reviews` — the aggregated recent guest reviews ("what guests say")
2. **Tokenizes** each document with the embedding model's own tokenizer and
   splits anything over the model's input limit into overlapping token windows
   (chunks)
3. **Embeds** every chunk (384-dim, normalized)
4. Writes a `documents` Delta table: one row per chunk — text, embedding, and
   the filterable listing fields (price, region, room type, MRT distance, …)
5. Creates a Vector Search endpoint + Delta Sync index over that table

Not wired to the web app — search is intentionally not connected yet.

**Model:** `intfloat/multilingual-e5-small` — reviews mix English, Chinese,
Spanish, and other languages, so an English-only model would handle a chunk of
them poorly. 384 dimensions, 512-token limit.

**Check before running**
- The cluster needs to download the model from huggingface.co. If your
  compute blocks outbound internet, the model load in section 1 will fail — tell
  me and we'll switch to a Databricks-hosted embedding endpoint or a model
  uploaded to a Volume.
- Vector Search availability and endpoint limits depend on your workspace
  plan — check under **Compute → Vector Search** that you can create an
  endpoint.
- ~2,900 chunks: fine on CPU (a few minutes); faster with a GPU.

In [ ]:
%pip install -q sentence-transformers databricks-vectorsearch

In [ ]:
dbutils.library.restartPython()

## Config

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog (blank = current default)")
dbutils.widgets.text("schema", "booking_system", "Schema name (same as import_to_delta)")
dbutils.widgets.text("vs_endpoint", "booking_vs_endpoint", "Vector Search endpoint name")

catalog = dbutils.widgets.get("catalog") or spark.sql("SELECT current_catalog()").first()[0]
schema = dbutils.widgets.get("schema")
VS_ENDPOINT = dbutils.widgets.get("vs_endpoint")

MODEL_NAME = "intfloat/multilingual-e5-small"
EMBEDDING_DIM = 384
MAX_TOKENS = 480     # chunk size in tokens; leaves headroom under the 512 limit for the
                     # "passage: " prefix and the 2 special tokens
OVERLAP = 48         # tokens shared between consecutive chunks of one document

def tbl(name):
    return f"{catalog}.{schema}.{name}"

DOCUMENTS_TABLE = tbl("documents")
INDEX_NAME = tbl("documents_index")
print(f"Reading/writing in {catalog}.{schema}; index = {INDEX_NAME}")

## 1. Load the model and its tokenizer

e5 models are trained with prefixes: `"passage: "` on documents,
`"query: "` on search queries. Both sides must be normalized. **Any future
query embedding must use this same model, the `query: ` prefix, and
normalization** — otherwise results degrade silently.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)
model.max_seq_length = 512

tokenizer = model.tokenizer
tokenizer.model_max_length = int(1e9)   # we chunk ourselves; silences the length warning

assert model.get_sentence_embedding_dimension() == EMBEDDING_DIM
print(f"device={device}, dim={EMBEDDING_DIM}")

# what tokenization looks like
sample = "quiet studio near MRT under $200"
print(sample, "->", tokenizer.tokenize(sample))

## 2. Build the documents

- `listing` document: name, property/room type, neighbourhood + region,
  description, amenities
- `reviews` document: the aggregated review text (listings with no reviews
  get none)

In [ ]:
import pandas as pd

lf = (spark.table(tbl("listing_features"))
      .select("id", "name", "property_type", "room_type", "neighbourhood_cleansed",
              "neighbourhood_group_cleansed", "description", "amenities")
      .toPandas())

rt = (spark.table(tbl("review_text"))
      .select("listing_id", "review_text", "reviews_used")
      .toPandas())
rt = rt[rt["review_text"].fillna("").str.len() > 0]

def listing_text(r):
    place = ", ".join(x for x in [r["neighbourhood_cleansed"], r["neighbourhood_group_cleansed"]]
                      if isinstance(x, str))
    parts = [f"{r['name']}. {r['property_type']} ({r['room_type']}) in {place}."]
    if isinstance(r["description"], str):
        parts.append(r["description"])
    if isinstance(r["amenities"], str):
        parts.append(f"Amenities: {r['amenities']}")
    return " ".join(parts)

docs = pd.concat([
    pd.DataFrame({"listing_id": lf["id"], "doc_type": "listing",
                  "text": lf.apply(listing_text, axis=1)}),
    pd.DataFrame({"listing_id": rt["listing_id"], "doc_type": "reviews",
                  "text": rt["review_text"]}),
], ignore_index=True)

print(docs["doc_type"].value_counts())

## 3. Tokenize and chunk

Each document is tokenized; if it fits in `MAX_TOKENS` it stays whole,
otherwise it is cut into overlapping token windows so no text is dropped by
truncation.

In [ ]:
def chunk_by_tokens(text, max_tokens=MAX_TOKENS, overlap=OVERLAP):
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    if len(ids) <= max_tokens:
        return [(text, len(ids))]
    step = max_tokens - overlap
    chunks = []
    for start in range(0, len(ids), step):
        window = ids[start:start + max_tokens]
        chunks.append((tokenizer.decode(window), len(window)))
        if start + max_tokens >= len(ids):
            break
    return chunks

rows = []
for r in docs.itertuples(index=False):
    for chunk_idx, (text, n_tokens) in enumerate(chunk_by_tokens(r.text)):
        rows.append({
            "doc_id": f"{r.listing_id}-{r.doc_type}-{chunk_idx}",
            "listing_id": r.listing_id,
            "doc_type": r.doc_type,
            "chunk_idx": chunk_idx,
            "n_tokens": n_tokens,
            "text": text,
        })
chunks = pd.DataFrame(rows)

print(f"{len(docs):,} documents -> {len(chunks):,} chunks")
print(chunks.groupby("doc_type")["n_tokens"].describe().round(1))
assert chunks["doc_id"].is_unique
assert chunks["n_tokens"].max() <= MAX_TOKENS

## 4. Embed

In [ ]:
embeddings = model.encode(
    ["passage: " + t for t in chunks["text"]],
    batch_size=32,
    normalize_embeddings=True,     # unit vectors: Vector Search ranks by L2 distance,
                                   # which then orders results identically to cosine
    show_progress_bar=True,
)
print(embeddings.shape)
chunks["embedding"] = embeddings.tolist()

## 5. Write the `documents` table

Chunk text + embedding, joined with the listing fields you'll want to
filter on (price, region, room type, capacity, rating, distances to
nearby places). Change Data Feed is enabled because a Delta Sync index
requires it on its source table.

In [ ]:
from pyspark.sql.functions import col

chunks_sdf = (spark.createDataFrame(chunks)
              .withColumn("embedding", col("embedding").cast("array<float>")))

meta = (spark.table(tbl("listing_features"))
        .select(col("id").alias("listing_id"), "price", "neighbourhood_cleansed",
                "neighbourhood_group_cleansed", "room_type", "property_type",
                "accommodates", "bedrooms", "review_scores_rating", "has_reviews",
                "minimum_nights", "host_is_superhost")
        .join(spark.table(tbl("listing_proximity")).select(
                  "listing_id", "mrt_station_nearest_km", "convenience_store_nearest_km",
                  "supermarket_nearest_km", "mall_nearest_km", "attraction_nearest_km"),
              on="listing_id", how="left"))

documents = chunks_sdf.join(meta, on="listing_id", how="left")

(documents.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(DOCUMENTS_TABLE))
spark.sql(f"ALTER TABLE {DOCUMENTS_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

print(f"{DOCUMENTS_TABLE}: {spark.table(DOCUMENTS_TABLE).count():,} rows")

## 6. Vector Search endpoint and index

Creating the endpoint the first time can take several minutes.

In [ ]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

existing_endpoints = [e["name"] for e in vsc.list_endpoints().get("endpoints", [])]
if VS_ENDPOINT not in existing_endpoints:
    vsc.create_endpoint_and_wait(name=VS_ENDPOINT, endpoint_type="STANDARD")
print(f"endpoint ready: {VS_ENDPOINT}")

In [ ]:
try:
    index = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=INDEX_NAME)
    index.sync()        # index already exists: pick up the rewritten table
    print("existing index: sync triggered")
except Exception:
    index = vsc.create_delta_sync_index_and_wait(
        endpoint_name=VS_ENDPOINT,
        index_name=INDEX_NAME,
        source_table_name=DOCUMENTS_TABLE,
        pipeline_type="TRIGGERED",
        primary_key="doc_id",
        embedding_dimension=EMBEDDING_DIM,
        embedding_vector_column="embedding",
    )
    print("index created")

## 7. Sanity check

A retrieval check only, to confirm the index answers — not the search
feature. Note the `query: ` prefix and normalization, and how the structured
filter (`price <= 200`) is applied alongside the vector match.

In [ ]:
query = "quiet place close to the MRT"
q_vec = model.encode(["query: " + query], normalize_embeddings=True)[0].tolist()

res = index.similarity_search(
    query_vector=q_vec,
    columns=["doc_id", "listing_id", "doc_type", "price", "neighbourhood_group_cleansed", "text"],
    filters={"price <=": 200},
    num_results=5,
)

for row in res["result"]["data_array"]:
    doc_id, listing_id, doc_type, price, region, text = row[:6]
    print(f"{doc_id} | {doc_type} | ${price} | {region}\n   {text[:140]}...\n")

## Notes for the query side (later)

- Embed queries with **the same model**, the **`query: ` prefix**, and
  `normalize_embeddings=True`.
- The index holds **chunks**. A listing can match through several chunks
  (its description chunk and/or one or more review chunks) — group results
  by `listing_id` and take the best score when you build the search response.
- `doc_type` can be filtered (`"doc_type": "listing"` vs `"reviews"`) to
  search only what hosts say or only what guests say.
- Re-running this notebook rewrites `documents` and re-syncs the index.